# Setup ENSEMBL annotations
##
### Author: Martin Loza
### Date: 25/11/20

Let's setup the ENSEMBL annotation for a given species for further use

In [21]:
# Change R language to English
Sys.setenv(LANGUAGE = "en")

# Init
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
})

# Local variables
seed = 777
date = "251120"

species = "human"
in_dir = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/ENSEMBL/raw/"

### Load the data

In [28]:
# Load the transcripts data
# Search for the file related to the species
files <- list.files(in_dir)
file_name <- files[grep(species, files)]
# paste0(in_dir,file_name)
# Load the transcripts
transcripts = read.table(paste0(in_dir,file_name),sep = "\t",header = TRUE,stringsAsFactors = FALSE)
head(transcripts,3)
colnames(transcripts)
cat("Number of transcripts loaded:", nrow(transcripts), "\n")

,Gene.stable.ID,Gene.stable.ID.version,Transcript.stable.ID,Transcript.stable.ID.version,Chromosome.scaffold.name,Gene.start..bp.,Gene.end..bp.,Strand,Transcript.start..bp.,Transcript.end..bp.,Transcription.start.site..TSS.,Gene.description,Gene.name,Version..transcript.,Version..gene.,Transcript.type,Gene.type,Gene...GC.content
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<dbl>
1,ENSG00000210049,ENSG00000210049.1,ENST00000387314,ENST00000387314.1,MT,577,647,1,577,647,577,mitochondrially encoded tRNA-Phe (UUU/C) [Source:HGNC Symbol;Acc:HGNC:7481],MT-TF,1,1,Mt_tRNA,Mt_tRNA,40.85
2,ENSG00000211459,ENSG00000211459.2,ENST00000389680,ENST00000389680.2,MT,648,1601,1,648,1601,648,mitochondrially encoded 12S rRNA [Source:HGNC Symbol;Acc:HGNC:7470],MT-RNR1,2,2,Mt_rRNA,Mt_rRNA,45.49
3,ENSG00000210077,ENSG00000210077.1,ENST00000387342,ENST00000387342.1,MT,1602,1670,1,1602,1670,1602,mitochondrially encoded tRNA-Val (GUN) [Source:HGNC Symbol;Acc:HGNC:7500],MT-TV,1,1,Mt_tRNA,Mt_tRNA,42.03


[1] "Gene.stable.ID"                 "Gene.stable.ID.version"        
 [3] "Transcript.stable.ID"           "Transcript.stable.ID.version"  
 [5] "Chromosome.scaffold.name"       "Gene.start..bp."               
 [7] "Gene.end..bp."                  "Strand"                        
 [9] "Transcript.start..bp."          "Transcript.end..bp."           
[11] "Transcription.start.site..TSS." "Gene.description"              
[13] "Gene.name"                      "Version..transcript."          
[15] "Version..gene."                 "Transcript.type"               
[17] "Gene.type"                      "Gene...GC.content"

Number of transcripts loaded: 230433 


### Handle transcript versions and create final annotations

We need to:
1. Extract transcript versions from the Transcript.stable.ID.version column
2. Keep only the highest version for each transcript ID
3. Create separate gene and transcript annotation files

#### 1. Extract transcript versions 

In [26]:
# We first confirm that the Transcript.stable.ID.version column exists
if(!"Transcript.stable.ID.version" %in% colnames(transcripts)) {
    stop("The column 'Transcript.stable.ID.version' does not exist in the transcripts data.")
} else {
    # Extract version number from the transcript ID
    transcripts$transcript_version <- as.numeric(str_extract(transcripts$Transcript.stable.ID.version, "(?<=\\.)\\d+$"))

    # Check for any NA values in transcript_version
    cat("Number of transcripts with missing version:", sum(is.na(transcripts$transcript_version)), "\n")

    # Show some examples of version extraction
    cat("Examples of transcript ID and extracted versions:\n")
    sample_idx <- sample(nrow(transcripts), 3)
    print(transcripts[sample_idx, c("Transcript.stable.ID", "Transcript.stable.ID.version", "transcript_version")])

    # Check the range of versions
    cat("\nTranscript version summary:\n")
    print(summary(transcripts$transcript_version))
}

Number of transcripts with missing version: 0 
Examples of transcript ID and extracted versions:
       Transcript.stable.ID Transcript.stable.ID.version transcript_version
147924      ENST00000743052            ENST00000743052.1                  1
188526      ENST00000461631            ENST00000461631.1                  1
87967       ENST00000809879            ENST00000809879.1                  1

Transcript version summary:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  1.000   1.000   1.000   1.935   1.000  17.000 


#### 2. Keep only the highest transcript versions

In [27]:
# check if the transcript_version columns exists
if(!"transcript_version" %in% colnames(transcripts)) {
    stop("The column 'transcript_version' does not exist in the transcripts data.")
} else{
    # Keep only the highest version for each transcript ID
    cat("Original number of transcripts:", nrow(transcripts), "\n")

    # Group by transcript stable ID and keep only the row with the highest version
    transcripts_filtered <- transcripts %>%
        group_by(Transcript.stable.ID) %>%
        slice_max(transcript_version, n = 1, with_ties = FALSE) %>%
        ungroup()

    cat("Number of transcripts after keeping highest versions:", nrow(transcripts_filtered), "\n")
    cat("Number of transcripts removed:", nrow(transcripts) - nrow(transcripts_filtered), "\n")

    # Check if we have any duplicated transcript IDs (should be 0)
    cat("Number of duplicated transcript IDs after filtering:", sum(duplicated(transcripts_filtered$Transcript.stable.ID)), "\n")
}

Original number of transcripts: 230433 
Number of transcripts after keeping highest versions: 230433 
Number of transcripts removed: 0 
Number of duplicated transcript IDs after filtering: 0 


Let's focus on protein coding genes and typical ncRNA genes

In [35]:
colnames(transcripts)

[1] "Gene.stable.ID"                 "Gene.stable.ID.version"        
 [3] "Transcript.stable.ID"           "Transcript.stable.ID.version"  
 [5] "Chromosome.scaffold.name"       "Gene.start..bp."               
 [7] "Gene.end..bp."                  "Strand"                        
 [9] "Transcript.start..bp."          "Transcript.end..bp."           
[11] "Transcription.start.site..TSS." "Gene.description"              
[13] "Gene.name"                      "Version..transcript."          
[15] "Version..gene."                 "Transcript.type"               
[17] "Gene.type"                      "Gene...GC.content"

In [ ]:
# Let's focus on protein coding genes and typical ncRNA genes
sel_genes <- c('protein_coding', 'lncRNA',
            'miRNA','misc_RNA','snRNA','rRNA','rRNA_pseudogene','snoRNA','scaRNA','sRNA',
            'pseudogene','unitary_pseudogene',
            'unprocessed_pseudogene','processed_pseudogene',
            'transcribed_unitary_pseudogene','transcribed_processed_pseudogene','translated_processed_pseudogene','transcribed_unprocessed_pseudogene',
            'IG_C_pseudogene','TR_V_pseudogene','IG_V_pseudogene','IG_J_pseudogene')

# we would like to get unique annotation for each transcript
transcripts_annotation <- transcripts_filtered %>%
  select(Chromosome.scaffold.name, Gene.start..bp., Gene.end..bp., Strand,
         Gene.stable.ID, Gene.stable.ID,
         Transcript.stable.ID, Transcript.start..bp., Transcript.end..bp., Transcript.type,
         Gene.type, Gene.name, Gene.description) %>%
  distinct() %>%
  # Calculate TSS based on strand: positive strand uses start, negative strand uses end
  mutate(TSS = ifelse(Strand == 1, Gene.start..bp., Gene.end..bp.)) %>%
  # Create priority for gene types (lower number = higher priority)
  mutate(gene_type_priority = case_when(
    Transcript.type %in% sel_genes ~ match(Transcript.type, sel_genes),
    TRUE ~ length(sel_genes)
    TRUE ~ 100  # All other types get lowest priority
  )) %>%
  # For each gene ID, keep only the one with highest priority (lowest number)
  group_by(Gene.stable.ID) %>%
  slice_min(gene_type_priority, n = 1, with_ties = FALSE) %>%
  ungroup() %>%
  # Remove the priority column as it's no longer needed
  select(-gene_type_priority) %>%
  arrange(Chromosome.scaffold.name, Gene.start..bp.)

[1] 22

[1] 22

In [41]:
any(duplicated(transcripts_filtered$Transcript.stable.ID))

[1] FALSE

In [40]:
transcripts_filtered %>%
  select(Chromosome.scaffold.name, Gene.start..bp., Gene.end..bp., Strand,
         Gene.stable.ID, Gene.stable.ID,
         Transcript.stable.ID, Transcript.start..bp., Transcript.end..bp., Transcript.type,
         Gene.type, Gene.name, Gene.description) %>% arrange(Transcript.stable.ID) %>% head(20)

Chromosome.scaffold.name,Gene.start..bp.,Gene.end..bp.,Strand,Gene.stable.ID,Transcript.stable.ID,Transcript.start..bp.,Transcript.end..bp.,Transcript.type,Gene.type,Gene.name,Gene.description
<chr>,<int>,<int>,<int>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>
12,8940358,8951856,-1,ENSG00000003056,ENST00000000412,8940361,8949645,protein_coding,protein_coding,M6PR,"mannose-6-phosphate receptor, cation dependent [Source:HGNC Symbol;Acc:HGNC:6752]"
12,2794290,2805423,1,ENSG00000004478,ENST00000001008,2794970,2805423,protein_coding,protein_coding,FKBP4,FKBP prolyl isomerase 4 [Source:HGNC Symbol;Acc:HGNC:3720]
2,72129238,72147862,-1,ENSG00000003137,ENST00000001146,72129238,72147862,protein_coding,protein_coding,CYP26B1,cytochrome P450 family 26 subfamily B member 1 [Source:HGNC Symbol;Acc:HGNC:20581]
6,143493961,143511842,-1,ENSG00000001036,ENST00000002165,143494812,143511720,protein_coding,protein_coding,FUCA2,alpha-L-fucosidase 2 [Source:HGNC Symbol;Acc:HGNC:4008]
16,90004871,90019890,-1,ENSG00000003249,ENST00000002501,90004871,90019456,protein_coding,protein_coding,DBNDD1,dysbindin domain containing 1 [Source:HGNC Symbol;Acc:HGNC:28455]
3,50155012,50189075,1,ENSG00000001617,ENST00000002829,50155324,50189075,protein_coding,protein_coding,SEMA3F,semaphorin 3F [Source:HGNC Symbol;Acc:HGNC:10728]
7,92084987,92134803,-1,ENSG00000001630,ENST00000003100,92112153,92134477,protein_coding,protein_coding,CYP51A1,cytochrome P450 family 51 subfamily A member 1 [Source:HGNC Symbol;Acc:HGNC:2649]
1,24355886,24416934,-1,ENSG00000001460,ENST00000003583,24357005,24413725,protein_coding,protein_coding,STPG1,sperm tail PG-rich repeat containing 1 [Source:HGNC Symbol;Acc:HGNC:28070]
7,150799598,150805122,1,ENSG00000002933,ENST00000004103,150800769,150805118,protein_coding,protein_coding,TMEM176A,transmembrane protein 176A [Source:HGNC Symbol;Acc:HGNC:24930]


In [32]:
unique(transcripts_filtered$Transcript.type)

[1] "protein_coding"                     "retained_intron"                   
 [3] "nonsense_mediated_decay"            "protein_coding_CDS_not_defined"    
 [5] "lncRNA"                             "unprocessed_pseudogene"            
 [7] "transcribed_unitary_pseudogene"     "processed_pseudogene"              
 [9] "transcribed_processed_pseudogene"   "transcribed_unprocessed_pseudogene"
[11] "unitary_pseudogene"                 "IG_V_gene"                         
[13] "protein_coding_LoF"                 "non_stop_decay"                    
[15] "miRNA"                              "misc_RNA"                          
[17] "snRNA"                              "rRNA"                              
[19] "rRNA_pseudogene"                    "snoRNA"                            
[21] "ribozyme"                           "TR_V_gene"                         
[23] "Mt_tRNA"                            "Mt_rRNA"                           
[25] "IG_C_gene"                          "IG_J_gene"                         
[27] "TR_J_gene"                          "IG_C_pseudogene"                   
[29] "TR_V_pseudogene"                    "IG_V_pseudogene"                   
[31] "TR_C_gene"                          "TEC"                               
[33] "scaRNA"                             "translated_processed_pseudogene"   
[35] "IG_D_gene"                          "sRNA"                              
[37] "pseudogene"                         "IG_J_pseudogene"                   
[39] "processed_transcript"               "TR_D_gene"